# 实验二：FASTSim 整车能耗仿真

## 实验目标
- 理解整车能耗仿真的基本原理（动力学 + 效率 MAP）
- 用 FASTSim 加载真实车型，跑 UDDS 城市工况
- 对比不同车型的百公里电耗

## 背景
FASTSim 是 NREL 开发的整车仿真工具，内置 20+ 真实车型参数。本实验跑一条标准城市驾驶循环，看电耗差异。

⚠️ 本 notebook 请用 **Python 3.10 (FASTSim)** 内核运行。

In [ ]:
import fastsim
print('FASTSim', fastsim.__version__)
from fastsim import cycle
import matplotlib
print('matplotlib', matplotlib.__version__)
import matplotlib.pyplot as plt

## 1. 加载循环工况
UDDS：美国城市驾驶循环，模拟市区走走停停。

In [ ]:
cyc = cycle.Cycle.from_file('udds')
print('时长: %.0f s, 里程: %.1f km' % (cyc.time_s[-1], cyc.dist_m[-1]/1000))

plt.figure(figsize=(10, 3))
plt.plot(cyc.time_s, cyc.mps * 3.6)
plt.xlabel('时间 (s)'); plt.ylabel('车速 (km/h)')
plt.title('UDDS 城市工况车速曲线')
plt.grid(True, alpha=0.3)
plt.show()

## 2. 加载车型并仿真
从 FASTSim 内置车型库选一款纯电动车。

In [ ]:
from fastsim.vehicle import Vehicle

# 列出可用车型（取前 5 个纯电）
import pandas as pd
veh_db = pd.read_csv(fastsim.resources.vehicles_db_path)
bevs = veh_db[veh_db['Powertrain'].str.contains('BEV', na=False)]
print(bevs['Vehicle'].head(5).to_string())

veh_name = bevs['Vehicle'].iloc[0]
veh = Vehicle.from_file(veh_name)
print('\n加载车型:', veh_name)

In [ ]:
# 跑仿真
sim = fastsim.Sim(veh, cyc)
sim.run()

# 百公里电耗
kwh_per_100km = sim.ess_kwh_out_ach[0] / (cyc.dist_m[-1]/1000) * 100
print('百公里电耗: %.1f kWh/100km' % kwh_per_100km)

## 3. 对比两款车型

In [ ]:
results = {}
for i in [0, 1]:
    vn = bevs['Vehicle'].iloc[i]
    v = Vehicle.from_file(vn)
    s = fastsim.Sim(v, cyc); s.run()
    e = s.ess_kwh_out_ach[0] / (cyc.dist_m[-1]/1000) * 100
    results[vn] = e
    print('%-30s %.1f kWh/100km' % (vn, e))

plt.figure(figsize=(8, 4))
plt.bar(range(len(results)), list(results.values()))
plt.xticks(range(len(results)), [k[:18] for k in results.keys()], rotation=15)
plt.ylabel('百公里电耗 (kWh/100km)')
plt.title('两款纯电动车 UDDS 工况能耗对比')
plt.tight_layout(); plt.show()

## 思考题
1. 为什么城市工况下电耗比高速工况低？（提示：动能回收）
2. 把车重增加 200kg，电耗变化多少？试着改 `veh.veh_kg` 再跑一次。
3. 如果换成 HWFET 高速工况，结果会反转吗？